# 第六章：Black-Scholes 期权定价 / Chapter 6: Black-Scholes Option Pricing

## 你将学到 / What You Will Learn
- 期权的价格是怎么算出来的 / How an option price is computed
- Black-Scholes 公式（诺贝尔奖级别的公式！）/ The Black-Scholes formula (a Nobel-prize-winning equation!)
- 用 QuantLib（专业金融计算库）来算期权价格 / Pricing options with QuantLib, a professional quant library
- 什么是期权的"3D价格曲面" / What an option's "3D price surface" looks like

---

## 6.1 期权的价格由什么决定？ / 6.1 What Drives the Price of an Option?

回想上一章的买房例子：那个"定金"应该是多少呢？

Back to the house-deposit analogy from the last chapter: how big should the deposit be?

想想看，哪些因素会影响这个定金的大小：

Several factors push it up or down:

| 因素 / Factor | 影响 / Effect | 直觉理解 / Intuition |
|------|------|---------|
| **当前房价 S / Spot S** | 越高→Call越贵 / Higher → call costlier | 现在就值钱，买入的权利当然更值钱 / If it is valuable today, the right to buy it is also valuable |
| **约定价格 K / Strike K** | 越高→Call越便宜 / Higher → call cheaper | 约定价太高，用到这个权利的机会就小 / A sky-high strike is unlikely to ever be useful |
| **剩余时间 T / Time T** | 越长→越贵 / Longer → costlier | 时间越长，变化空间越大 / More time means more room for the price to move |
| **市场波动 σ / Volatility σ** | 越大→越贵 / Higher → costlier | 波动越大，赚大钱的机会越多 / Wilder swings → bigger payoff opportunities |
| **利率 r / Rate r** | 越高→Call越贵 / Higher → call costlier | 金融学上的折现效应 / Discounting effect from finance theory |

## 6.2 Black-Scholes 公式 / 6.2 The Black-Scholes Formula

1973年，三位经济学家推导出了一个公式，可以精确计算欧式期权的价格：

In 1973 three economists derived a formula that prices European options exactly:

$$\text{Call} = S \cdot N(d_1) - K \cdot e^{-rT} \cdot N(d_2)$$

$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2) \cdot T}{\sigma \cdot \sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

> 其中 $N(x)$ 是正态分布的累积函数（不用手算，Python帮你算）
>
> $N(x)$ is the cumulative standard-normal distribution — you never compute it by hand; Python does it for you.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
from scipy.stats import norm
try:
    import QuantLib as ql
    HAS_QL = True
    print(f'QuantLib version: {ql.__version__}')
except ImportError:
    HAS_QL = False
    print('QuantLib not installed; falling back to scipy BS only.')
print('Setup done!')

## 6.3 互动定价器 / 6.3 Interactive Pricer

> 拖动五个滑块，观察：
> - 左图：Call/Put价格随标的价格的变化
> - 中图：3D价格曲面（标的价+波动率→期权价格）
> - 右图：Black-Scholes 计算结果
>
> Drag the five sliders and watch:
> - Left chart: call / put prices vs. spot
> - Middle chart: 3D price surface (spot + vol → option price)
> - Right chart: the Black-Scholes read-out

In [ ]:
def bs_price(S, K, T, r, sigma, opt='call'):
    d1 = (np.log(S / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if opt == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        delta = norm.cdf(d1)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        delta = norm.cdf(d1) - 1
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega = S * norm.pdf(d1) * np.sqrt(T) / 100  # per 1% vol
    theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
             - r * K * np.exp(-r * T) * (norm.cdf(d2) if opt == 'call' else -norm.cdf(-d2))) / 365
    return {'price': price, 'delta': delta, 'gamma': gamma, 'vega': vega, 'theta': theta}


def bs_pricer_ui(S=100, K=100, T=1.0, r_pct=5.0, sigma_pct=20.0):
    plt.close('all')
    r = r_pct / 100
    sigma = sigma_pct / 100

    res = {'Call': bs_price(S, K, T, r, sigma, 'call'),
           'Put':  bs_price(S, K, T, r, sigma, 'put')}

    fig = plt.figure(figsize=(15, 5))

    # Left: price curves
    ax1 = fig.add_subplot(131)
    S_range = np.linspace(S * 0.5, S * 1.5, 100)
    d1 = (np.log(S_range / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_curve = S_range * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    put_curve = K * np.exp(-r * T) * norm.cdf(-d2) - S_range * norm.cdf(-d1)

    ax1.plot(S_range, call_curve, color=C_GREEN, lw=2.5, label=f"Call = ${res['Call']['price']:.2f}")
    ax1.plot(S_range, put_curve, color=C_RED, lw=2.5, label=f"Put = ${res['Put']['price']:.2f}")
    ax1.plot(S_range, np.maximum(S_range - K, 0), C_GREEN, ls='--', alpha=0.3, label='Intrinsic')
    ax1.plot(S_range, np.maximum(K - S_range, 0), C_RED, ls='--', alpha=0.3)
    ax1.plot(S, res['Call']['price'], 'go', ms=10, zorder=5)
    ax1.plot(S, res['Put']['price'], 'ro', ms=10, zorder=5)
    ax1.axvline(x=K, color=C_PURPLE, ls=':', alpha=0.5)
    ax1.set_xlabel('Spot S'); ax1.set_ylabel('Option price')
    ax1.set_title('Option Price Curves', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=9); ax1.grid(True, alpha=0.2)

    # Middle: 3D surface
    ax2 = fig.add_subplot(132, projection='3d')
    Sr = np.linspace(S * 0.6, S * 1.4, 30)
    Vr = np.linspace(0.05, 0.60, 30)
    Sg, Vg = np.meshgrid(Sr, Vr)
    d1g = (np.log(Sg / K) + (r + Vg**2 / 2) * T) / (Vg * np.sqrt(T))
    d2g = d1g - Vg * np.sqrt(T)
    Cg = Sg * norm.cdf(d1g) - K * np.exp(-r * T) * norm.cdf(d2g)
    ax2.plot_surface(Sg, Vg * 100, Cg, cmap=cm.viridis, alpha=0.8)
    ax2.set_xlabel('Spot S', fontsize=8)
    ax2.set_ylabel('Vol sigma (%)', fontsize=8)
    ax2.set_zlabel('Call price', fontsize=8)
    ax2.set_title('3D Price Surface', fontweight='bold', fontsize=10)

    # Right: results panel
    ax3 = fig.add_subplot(133)
    ax3.set_xlim(0, 10); ax3.set_ylim(0, 10); ax3.axis('off')
    ax3.set_title('Black-Scholes Results', fontsize=13, fontweight='bold')
    ax3.text(2, 9.3, 'Metric', fontsize=9, fontweight='bold')
    ax3.text(5.5, 9.3, 'Call', fontsize=9, fontweight='bold', color=C_GREEN)
    ax3.text(8.5, 9.3, 'Put', fontsize=9, fontweight='bold', color=C_RED)
    ax3.plot([0.5, 9.5], [9.0, 9.0], color='gray', lw=0.5)
    metrics = [('Price', 'price'),
               ('Delta (direction)', 'delta'),
               ('Gamma (curvature)', 'gamma'),
               ('Theta (time decay)', 'theta'),
               ('Vega (vol sens.)', 'vega')]
    for i, (label, key) in enumerate(metrics):
        y = 8 - i * 1.5
        ax3.text(2, y, label, fontsize=9, ha='center')
        ax3.text(5.5, y, f"{res['Call'][key]:.4f}", fontsize=10, ha='center', fontweight='bold', color=C_GREEN)
        ax3.text(8.5, y, f"{res['Put'][key]:.4f}", fontsize=10, ha='center', fontweight='bold', color=C_RED)

    plt.tight_layout()
    plt.show()

interact(bs_pricer_ui,
         S=FloatSlider(value=100, min=50, max=200, step=1, description='Spot S:'),
         K=FloatSlider(value=100, min=50, max=200, step=1, description='Strike K:'),
         T=FloatSlider(value=1.0, min=0.05, max=3.0, step=0.05, description='T (yrs):'),
         r_pct=FloatSlider(value=5.0, min=0, max=15, step=0.5, description='Rate r(%):'),
         sigma_pct=FloatSlider(value=20.0, min=5, max=80, step=1, description='Vol sigma(%):'));

## 6.4 小结 / 6.4 Chapter Recap

> 你不需要记住BS公式，但要记住：
>
> You do not need to memorize Black-Scholes, but you should remember the sign of each effect:
>
> | 参数变大 / Parameter Increases | Call价格 / Call | Put价格 / Put |
> |---------|---------|---------|
> | 标的价S ↑ / Spot S ↑ | 变贵 / costlier | 变便宜 / cheaper |
> | 行权价K ↑ / Strike K ↑ | 变便宜 / cheaper | 变贵 / costlier |
> | 时间T ↑ / Time T ↑ | 变贵 / costlier | 变贵 / costlier |
> | 波动率σ ↑ / Vol σ ↑ | 变贵 / costlier | 变贵 / costlier |
> | 利率r ↑ / Rate r ↑ | 变贵 / costlier | 变便宜 / cheaper |

> **下一章：希腊字母** —— 衡量期权对各参数的敏感程度
>
> **Next chapter: the Greeks** — precise sensitivities of the option price to each input.